# Environment check

Run every cell. If all three tests print results without errors, the machine is ready for the test.

If a package is missing, open **Anaconda Prompt** and run:

```
pip install pyomo simpy "ortools>=9.15,<9.16" "highspy>=1.11,<1.13"
```

(OR-Tools is not available from conda on Windows, and newer HiGHS versions clash with it.)

then restart the kernel and run this notebook again.

# Pyomo Test

In [ ]:
from pyomo.environ import *

# Model
model = ConcreteModel()

# Variable
model.x = Var(domain=NonNegativeReals)

# Objective
model.obj = Objective(expr=2 * model.x, sense=maximize)

# Constraint
model.con = Constraint(expr=model.x <= 5)

# Solve
solver = SolverFactory("appsi_highs")
solver.solve(model, tee=False)

# Display the model
model.pprint()

# Display the results
model.display()

# Google OR-Tools Test

In [ ]:
from ortools.sat.python import cp_model

"""Minimal CP-SAT example to showcase calling the solver."""
# Creates the model.
model = cp_model.CpModel()

# Creates the variables.
var_upper_bound = max(50, 45, 37)
x = model.new_int_var(0, var_upper_bound, "x")
y = model.new_int_var(0, var_upper_bound, "y")
z = model.new_int_var(0, var_upper_bound, "z")

# Creates the constraints.
model.add(2 * x + 7 * y + 3 * z <= 50)
model.add(3 * x - 5 * y + 7 * z <= 45)
model.add(5 * x + 2 * y - 6 * z <= 37)

model.maximize(2 * x + 2 * y + 3 * z)

# Creates a solver and solves the model.
solver = cp_model.CpSolver()
status = solver.solve(model)

if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print(f"Maximum of objective function: {solver.objective_value}\n")
    print(f"x = {solver.value(x)}")
    print(f"y = {solver.value(y)}")
    print(f"z = {solver.value(z)}")
else:
    print("No solution found.")

# Simpy Test

In [ ]:
import simpy

def simple_process(env):
    print("Start at", env.now)
    yield env.timeout(5)
    print("End at", env.now)

env = simpy.Environment()
env.process(simple_process(env))
env.run()